# Routing experiments: a social network for AI agents

Stage 1 only: setup, a sourced dataset, and routing experiments. No API app, database, worker, or dashboard is built here.

Source of truth: `PLAN_1_agent_social_network.md`, overridden by `docs/PLAN_AMENDMENTS.md`.

- Relevance means business complementarity, not matching company names.
- Colab secret: `AICREDITS_KEY`. Its value is never printed or saved.
- Session LLM spend cap: USD 1.50. Every paid call needs known prices, a token cap, and a usage log.
- The final model and routing thresholds require Anand's approval.

## Step 1: pinned installs

In [ ]:
import importlib.metadata
import random
import subprocess
import sys
from time import perf_counter

PINS = {
    "fastembed": "0.7.3", "openai": "1.109.1", "pandas": "2.2.3",
    "numpy": "2.2.6", "pydantic": "2.11.7", "onnxruntime": "1.22.1",
}
SEED = 42
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
EMBED_DIMENSIONS = 384
LLM_BASE_URL = "https://aicredits.in/v1"
SECRET_NAME = "AICREDITS_KEY"
SESSION_BUDGET_USD = 1.50
random.seed(SEED)
print(f"Python {sys.version.split()[0]}; seed={SEED}; configuration only.")

In [ ]:
def _installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

_missing = [f"{name}=={version}" for name, version in PINS.items() if _installed_version(name) != version]
_started = perf_counter()
try:
    if _missing:
        _install = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
             "--timeout", "10", "--retries", "0", *_missing],
            capture_output=True, text=True, timeout=50,
        )
        print("Pinned install passed:", _install.returncode == 0)
        if _install.returncode:
            raise RuntimeError("Pinned install failed; inspect the local pip output before continuing.")
    else:
        print("All six pins already installed; no installation request made.")
except subprocess.TimeoutExpired:
    print("TimeoutExpired: installation stopped at 50 seconds.")
    raise
finally:
    print(f"Elapsed seconds: {perf_counter() - _started:.3f}")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(SEED)
assert all(importlib.metadata.version(name) == version for name, version in PINS.items())
print(pd.DataFrame([{"package": name, "version": importlib.metadata.version(name)} for name in PINS]).to_string(index=False))

## Step 1: cache, secret access, and model catalogue

Run one small cell at a time. Each action is separate and bounded. Any human authorization must be completed manually before a fast verification.

1a is a manual-only secret read of `AICREDITS_KEY`, whose permission is already granted. Do not rerun it through MCP, because the Colab UI can time out. No key value is displayed or written.

1b checks Drive only. Drive has been verified as present. Persistent project caches use `/content/drive/MyDrive/agent-network-experiments/cache`. A runtime-only fallback requires explicit approval.

1c defines cache helpers without I/O. 1d makes one free timed catalogue request with a 20-second timeout, zero SDK retries and a 55-second cell deadline. 1e writes the catalogue and displays numeric per-million prices.

The later shortlist is a limited trial pool, not a final model choice. Paid calls must be prepared with known prices, a budget reservation and a cache check in a separate cell, then finalized to persistent response and usage records.

In [ ]:
from google.colab import userdata

_aicredits_key = None
try:
    _aicredits_key = userdata.get("AICREDITS_KEY")
    if not isinstance(_aicredits_key, str) or not _aicredits_key.strip():
        raise ValueError("Empty secret")
    print(True)
except Exception as error:
    _aicredits_key = None
    print(False)
    print(type(error).__name__)

In [ ]:
import subprocess
import sys
from time import perf_counter

_started = perf_counter()
try:
    _check = subprocess.run(
        [sys.executable, "-c", "from pathlib import Path\nprint(Path('/content/drive/MyDrive').is_dir())"],
        capture_output=True, text=True, timeout=5,
    )
    DRIVE_PRESENT = _check.returncode == 0 and _check.stdout.strip() == "True"
    print(DRIVE_PRESENT)
except subprocess.TimeoutExpired:
    DRIVE_PRESENT = False
    print(False)
    print("TimeoutExpired")
finally:
    print(f"Elapsed seconds: {perf_counter() - _started:.3f}")

In [ ]:
import hashlib
import json
from pathlib import Path

if not DRIVE_PRESENT and not globals().get("ALLOW_RUNTIME_CACHE", False):
    raise RuntimeError("Mount Drive manually or approve /content/cache first.")
CACHE_ROOT = Path("/content/drive/MyDrive/agent-network-experiments/cache" if DRIVE_PRESENT else "/content/cache")

def content_key(payload):
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def cache_read(namespace, key):
    path = CACHE_ROOT / namespace / f"{key}.json"
    return json.loads(path.read_text()) if path.is_file() else None

def cache_write(namespace, key, payload):
    path = CACHE_ROOT / namespace / f"{key}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, sort_keys=True))
    temporary.replace(path)

print("Cache helpers ready; no disk reads or writes performed.")

In [ ]:
import signal
from time import perf_counter
from openai import OpenAI

def _cell_deadline(*_):
    raise TimeoutError("Cell reached its 55-second limit.")

client = OpenAI(api_key=_aicredits_key, base_url="https://aicredits.in/v1", timeout=20.0, max_retries=0)
_previous_handler = signal.signal(signal.SIGALRM, _cell_deadline)
_started = perf_counter()
signal.setitimer(signal.ITIMER_REAL, 55.0)
try:
    catalogue = client.models.list().model_dump(mode="json")
    print(len(catalogue.get("data", [])))
except Exception as error:
    catalogue = None
    print(type(error).__name__)
finally:
    signal.setitimer(signal.ITIMER_REAL, 0.0)
    signal.signal(signal.SIGALRM, _previous_handler)
    print(f"Elapsed seconds: {perf_counter() - _started:.3f}")

In [ ]:
import signal
import pandas as pd
from datetime import datetime, timezone

_previous_handler = signal.signal(signal.SIGALRM, _cell_deadline)
signal.setitimer(signal.ITIMER_REAL, 55.0)
try:
    cache_write("model_catalogue", content_key({"base_url": "https://aicredits.in/v1", "operation": "models"}), {"fetched_at_utc": datetime.now(timezone.utc).isoformat(), "response": catalogue})
    _models = pd.DataFrame(catalogue["data"])
    print("Pricing fields present:", all(field in _models for field in ("input_cost_per_token", "output_cost_per_token")))
    _chat = _models.loc[_models["mode"].eq("chat")].copy()
    for direction in ("input", "output"):
        _chat[f"USD_{direction}_per_M"] = pd.to_numeric(_chat[f"{direction}_cost_per_token"], errors="coerce") * 1e6
        _chat[f"INR_{direction}_per_M"] = pd.to_numeric(_chat[f"{direction}_cost_per_token_inr"], errors="coerce") * 1e6
    _chat = _chat.dropna(subset=["USD_input_per_M", "USD_output_per_M"])
    _chat = _chat.loc[(_chat.USD_input_per_M >= 0) & (_chat.USD_output_per_M >= 0)]
    _chat["price_sum"] = _chat["USD_input_per_M"] + _chat["USD_output_per_M"]
    shortlist = _chat.sort_values("price_sum").head(3)
    print("Provisional shortlist: rates per million tokens, sorted by input+output USD rates.")
    display(shortlist[["id", "USD_input_per_M", "USD_output_per_M", "INR_input_per_M", "INR_output_per_M"]], include=["text/plain"])
except Exception as error:
    print(type(error).__name__)
finally:
    signal.setitimer(signal.ITIMER_REAL, 0.0)
    signal.signal(signal.SIGALRM, _previous_handler)

In [ ]:
# Price screening only. Use established instruction/chat families for this small trial.
_general_families = ("mistralai/", "meta-llama/", "google/", "qwen/", "deepseek/", "openai/", "anthropic/")
_eligible = _chat["id"].str.startswith(_general_families, na=False)
_task_specific = _chat["id"].str.contains(r"coder|codestral|codegen|embed|guard|moderation", case=False, regex=True)
_judge_pool = _chat.loc[_eligible & ~_task_specific].copy()
shortlist = _judge_pool.sort_values(["price_sum", "id"]).head(3)
assert len(shortlist) == 3
print("Provisional general chat candidates; judge quality is untested.")
_columns = ["id", "USD_input_per_M", "USD_output_per_M", "INR_input_per_M", "INR_output_per_M"]
print(shortlist[_columns].to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("ScaleDown utilities and roleplay merges are outside this trial pool.")
print("No API request made in this price-screening cell.")

In [ ]:
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone

STAGE1_STARTED_AT = datetime.fromtimestamp(1794715143.629, timezone.utc)
STAGE1_DEADLINE = STAGE1_STARTED_AT + timedelta(minutes=200)

def check_stage1_time():
    remaining = (STAGE1_DEADLINE - datetime.now(timezone.utc)).total_seconds()
    if remaining <= 0:
        raise TimeoutError("Stage 1 reached its original 3h20m cap. Stop and report.")
    return remaining

@contextmanager
def cell_limit(seconds=55.0):
    remaining = check_stage1_time()
    previous = signal.signal(signal.SIGALRM, _cell_deadline)
    signal.setitimer(signal.ITIMER_REAL, min(seconds, remaining))
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, previous)

print(f"Time guard defined; {check_stage1_time() / 60:.1f} minutes remain.")

In [ ]:
import math

RATES = {}
for _model in catalogue["data"]:
    _input = _model.get("input_cost_per_token")
    _output = _model.get("output_cost_per_token")
    if _model.get("mode") == "chat" and all(isinstance(rate, (int, float)) and math.isfinite(rate) and rate >= 0 for rate in (_input, _output)):
        RATES[_model["id"]] = {
            "input_usd": _input, "output_usd": _output,
            "input_inr": _model.get("input_cost_per_token_inr"),
            "output_inr": _model.get("output_cost_per_token_inr"),
        }
SESSION_TAG = "stage1-20261115-035903"
print("Known finite prices loaded for", len(RATES), "chat entries.")
print("Session spend cap USD:", SESSION_BUDGET_USD)

In [ ]:
with cell_limit():
    ledger = cache_read("usage_ledger", SESSION_TAG) or {"session_tag": SESSION_TAG, "calls": {}}
assert ledger["session_tag"] == SESSION_TAG and isinstance(ledger["calls"], dict)
MAX_PAID_CALLS = 240

def budget_used():
    return sum(float(row.get("charged_usd", row.get("reserved_usd", 0.0))) for row in ledger["calls"].values())

print("Logged paid requests:", len(ledger["calls"]))
print(f"Spent or conservatively reserved USD: {budget_used():.8f}")
print("Persistent ledger ready; no API request made.")

In [ ]:
def chat_plan(model_id, messages, purpose, max_tokens, trial_id=None):
    check_stage1_time()
    assert model_id in RATES, "Unknown prices: no paid call permitted."
    assert isinstance(max_tokens, int) and 1 <= max_tokens <= 700
    assert isinstance(messages, list) and 1 <= len(messages) <= 3
    request = {"model": model_id, "messages": messages, "temperature": 0, "max_tokens": max_tokens}
    if purpose == "smoke":
        request["response_format"] = {"type": "json_object"}
    rates = RATES[model_id]
    input_bound = len(json.dumps(messages, ensure_ascii=False).encode("utf-8")) + 128 * len(messages) + 512
    reserve_usd = input_bound * rates["input_usd"] + max_tokens * rates["output_usd"]
    key = content_key({"base_url": LLM_BASE_URL, "purpose": purpose, "request": request, "trial_id": trial_id})
    return {"key": key, "request": request, "purpose": purpose, "trial_id": trial_id,
            "rates": rates, "reserved_usd": reserve_usd, "input_bound": input_bound}

print("Pure request planner defined; token cap, prices and trial-aware cache key are mandatory.")

In [ ]:
def reserve_chat(plan):
    check_stage1_time()
    cached = cache_read("chat_responses", plan["key"])
    if cached is not None:
        return {**plan, "cached": True, "cached_result": cached}
    if plan["key"] in ledger["calls"]:
        raise RuntimeError("Unresolved or uncached earlier request: stop, do not retry automatically.")
    if len(ledger["calls"]) >= MAX_PAID_CALLS or budget_used() + plan["reserved_usd"] > SESSION_BUDGET_USD:
        raise RuntimeError("Paid-call budget or request-count cap reached.")
    row = {"purpose": plan["purpose"], "model": plan["request"]["model"],
           "trial_id": plan["trial_id"], "status": "reserved", "cached": False,
           "reserved_usd": plan["reserved_usd"], "max_tokens": plan["request"]["max_tokens"],
           "tokens_in": None, "tokens_out": None, "created_at_utc": datetime.now(timezone.utc).isoformat()}
    ledger["calls"][plan["key"]] = row
    cache_write("usage_ledger", SESSION_TAG, ledger)
    return {**plan, "cached": False, "attempted": False}

print("Cache lookup and persistent budget reservation defined; no API request made.")

In [ ]:
def run_chat(plan):
    check_stage1_time()
    if plan["cached"]:
        print("Response cache hit; no API request. Elapsed seconds: 0.000")
        return {"cached": True, "response": plan["cached_result"]["response"], "error": None, "elapsed_s": 0.0}
    if plan["attempted"]:
        raise RuntimeError("This prepared request was already attempted; do not replay it.")
    plan["attempted"] = True
    started = perf_counter()
    result = {"cached": False, "response": None, "error": None}
    try:
        with cell_limit():
            response = client.with_options(timeout=20.0, max_retries=0).chat.completions.create(**plan["request"])
        result["response"] = response.model_dump(mode="json")
    except Exception as error:
        result["error"] = type(error).__name__
        print(type(error).__name__, "HTTP", getattr(error, "status_code", "n/a"))
    finally:
        result["elapsed_s"] = perf_counter() - started
        print(f"Elapsed seconds: {result['elapsed_s']:.3f}")
    return result

print("One-request SDK runner defined. No API request made by this definition cell.")

In [ ]:
def priced_usage(plan, result):
    if result["cached"]:
        return {"cached": True, "charged_usd": 0.0, "elapsed_s": result["elapsed_s"]}
    row = dict(ledger["calls"][plan["key"]])
    usage = (result["response"] or {}).get("usage") or {}
    tokens_in, tokens_out = usage.get("prompt_tokens"), usage.get("completion_tokens")
    known = all(type(value) is int and value >= 0 for value in (tokens_in, tokens_out))
    rates = plan["rates"]
    cost = tokens_in * rates["input_usd"] + tokens_out * rates["output_usd"] if known else plan["reserved_usd"]
    inr_known = all(isinstance(rates[key], (int, float)) and math.isfinite(rates[key]) for key in ("input_inr", "output_inr"))
    cost_inr = tokens_in * rates["input_inr"] + tokens_out * rates["output_inr"] if known and inr_known else None
    row.update({"tokens_in": tokens_in, "tokens_out": tokens_out, "charged_usd": cost,
                "cost_estimate_inr": cost_inr, "elapsed_s": result["elapsed_s"], "error": result["error"],
                "status": "completed" if result["response"] else "failed_unknown_charge",
                "cost_basis": "reported_tokens_full_input_rate" if known else "conservative_reservation",
                "cap_violation": known and tokens_out > plan["request"]["max_tokens"]})
    return row

print("Usage pricing defined; missing usage keeps the conservative reservation.")

In [ ]:
def persist_chat(plan, result):
    row = priced_usage(plan, result)
    if not result["cached"]:
        if result["response"] is not None:
            cache_write("chat_responses", plan["key"], {"response": result["response"], "usage": row})
        ledger["calls"][plan["key"]] = row
        cache_write("usage_ledger", SESSION_TAG, ledger)
        if row["cap_violation"]:
            raise RuntimeError("Provider exceeded max_tokens; usage saved. Stop paid calls.")
    return row

print("Response and usage persistence defined separately from the network call.")

In [ ]:
_smoke_messages = [
    {"role": "system", "content": "Return JSON only: {\"ok\": true}."},
    {"role": "user", "content": "Connectivity test. Reply exactly {\"ok\": true}."},
]
with cell_limit():
    smoke_plan = reserve_chat(chat_plan("mistralai/mistral-nemo", _smoke_messages, "smoke", 32))
print("Cached response available:", smoke_plan["cached"])
print(f"Maximum conservatively reserved USD: {smoke_plan['reserved_usd']:.8f}")
print("No API request made in this preparation cell.")

In [ ]:
smoke_result = run_chat(smoke_plan)
print("Chat completion received:", smoke_result["response"] is not None)

In [ ]:
with cell_limit():
    smoke_usage = persist_chat(smoke_plan, smoke_result)
print("Response and usage persisted:", smoke_result["response"] is not None)
print("Input/output tokens:", smoke_usage.get("tokens_in"), smoke_usage.get("tokens_out"))
print(f"Estimated USD: {smoke_usage['charged_usd']:.8f}; session USD: {budget_used():.8f}")
if smoke_usage.get("cost_estimate_inr") is not None:
    print(f"Estimated INR: {smoke_usage['cost_estimate_inr']:.6f}")

In [ ]:
assert smoke_result["response"] is not None, "No completion to validate."
_smoke_text = smoke_result["response"]["choices"][0]["message"]["content"]
_smoke_json = json.loads(_smoke_text)
SMOKE_PASSED = isinstance(_smoke_json, dict) and set(_smoke_json) == {"ok"} and _smoke_json["ok"] is True
assert SMOKE_PASSED, "Smoke response did not match the required JSON."
print("Smoke JSON valid:", SMOKE_PASSED)
print("No final judge model or routing thresholds have been selected.")